# Notebook 10: XGBoost

**Difficulty:** ⭐⭐⭐⭐⭐ | **Estimated Time:** 3.5 hours | **Week 9 — Tree-Based Methods & Gradient Boosting**

---

## Why This Matters

XGBoost (eXtreme Gradient Boosting) has won more Kaggle competitions than any other single algorithm. It is the industry standard for tabular data: used in credit scoring, fraud detection, ad click-through rate prediction, and medical diagnosis. It takes the gradient boosting framework from Notebook 9 and makes it faster, smarter, and more regularized by exploiting second-order information. Understanding XGBoost means understanding why industry practitioners reach for it by default.

---

## Real-World Analogy: The Sculptor

Standard gradient boosting is like a sculptor who uses only the **direction** of the chisel (first derivative = gradient) to decide where to cut. They know which way to move, but not how hard the material is.

XGBoost is like a sculptor who uses both:
- The **chisel direction** (first derivative = gradient $g_i$): which way to cut
- The **curvature of the material** (second derivative = Hessian $h_i$): how hard or soft the stone is at that point

Knowing both direction AND curvature means each cut is more precise — you apply more force where the stone is soft (flat loss region) and less force where it's hard (steep region). This is why XGBoost needs fewer trees to achieve the same accuracy.

---

## Learning Objectives

By the end of this notebook you will be able to:
1. Derive the XGBoost objective from Taylor expansion principles
2. Explain optimal leaf weights and split gain formula
3. Demonstrate XGBoost's native missing value handling
4. Tune the key XGBoost hyperparameters systematically
5. Use early stopping to find the optimal number of trees automatically

## Section 1: Theory — What XGBoost Adds Over Gradient Boosting

### The Five Key Innovations

| Innovation | What It Does |
|:---|:---|
| **Regularized objective** | L1 + L2 penalties on leaf weights; prevents overfit without cross-validation |
| **Second-order Taylor expansion** | Uses gradient $g_i$ AND Hessian $h_i$ for better step sizes |
| **Histogram-based splitting** | Approximate splits via binning → 10–100× faster on large data |
| **Column subsampling** | Randomly sample features per tree/level (like random forests) |
| **Native missing value handling** | Learns optimal direction for missing values at each split |

### The XGBoost Objective

At each boosting step $m$, we minimize:
$$\tilde{\mathcal{L}}^{(m)} = \sum_i \left[ g_i f_m(x_i) + \frac{1}{2} h_i f_m(x_i)^2 \right] + \Omega(f_m)$$

Where:
- $g_i = \frac{\partial l(y_i, \hat{y}_i^{(m-1)})}{\partial \hat{y}_i^{(m-1)}}$ — **gradient** (first derivative of loss)
- $h_i = \frac{\partial^2 l(y_i, \hat{y}_i^{(m-1)})}{\partial (\hat{y}_i^{(m-1)})^2}$ — **Hessian** (second derivative of loss)
- $\Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_j w_j^2 + \alpha \sum_j |w_j|$ — **regularization** on $T$ leaves with weights $w_j$

### Optimal Leaf Weights

For a fixed tree structure with leaf $j$ containing samples $I_j$, the optimal leaf weight is:
$$w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}$$

This is a **closed-form solution** — no iterative optimization needed per tree!

### Split Gain Formula

The gain from splitting leaf $j$ into left ($L$) and right ($R$) children:
$$\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right] - \gamma$$

Where $G = \sum g_i$, $H = \sum h_i$ for left/right samples. A split is only made if Gain > 0. The $\gamma$ term is a **built-in pruning threshold**.

### For MSE Loss

With $l = \frac{1}{2}(y_i - \hat{y}_i)^2$:
- $g_i = \hat{y}_i - y_i$ (predicted minus actual)
- $h_i = 1$ (constant — the Hessian is flat for MSE!)

So the optimal leaf weight becomes: $w_j^* = -\frac{\sum_{i \in I_j} g_i}{n_j + \lambda} = \frac{\sum_{i \in I_j}(y_i - \hat{y}_i)}{n_j + \lambda}$

This is just the **mean residual** in that leaf, regularized by $\lambda$.

In [ ]:
# ── Setup & Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── XGBoost: import with graceful fallback ─────────────────────────────────
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print(f"XGBoost version {xgb.__version__} loaded successfully.")
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed. Falling back to sklearn GradientBoostingRegressor.")
    print("To install: pip install xgboost")
    print("Note: Missing-value handling demo requires XGBoost.")

    # Create a thin wrapper so the rest of the notebook runs unchanged
    class _XGBRegressor:
        """Thin sklearn-GBR shim that mimics XGBRegressor's API surface."""
        def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3,
                     subsample=1.0, colsample_bytree=1.0, reg_lambda=1.0,
                     reg_alpha=0.0, gamma=0.0, min_child_weight=1,
                     early_stopping_rounds=None, eval_metric=None,
                     random_state=42, **kwargs):
            self.n_estimators         = n_estimators
            self.learning_rate        = learning_rate
            self.max_depth            = max_depth
            self.subsample            = subsample
            self.random_state         = random_state
            self.early_stopping_rounds = early_stopping_rounds
            self._model = GradientBoostingRegressor(
                n_estimators=n_estimators, learning_rate=learning_rate,
                max_depth=max_depth, subsample=subsample, random_state=random_state
            )
            self.evals_result_ = {}

        def fit(self, X, y, eval_set=None, verbose=False, **kwargs):
            self._model.fit(X, y)
            if eval_set:
                X_val, y_val = eval_set[0]
                tr = [np.sqrt(mean_squared_error(y, p))
                      for p in self._model.staged_predict(X)]
                vl = [np.sqrt(mean_squared_error(y_val, p))
                      for p in self._model.staged_predict(X_val)]
                self.evals_result_ = {'validation_0': {'rmse': vl},
                                      'train':        {'rmse': tr}}
                self.best_iteration = int(np.argmin(vl))
            return self

        def predict(self, X):
            return self._model.predict(X)

    class _xgb_ns:
        XGBRegressor = _XGBRegressor

    xgb = _xgb_ns()

print("\nAll imports complete.")

## Section 2: House Price Dataset with Missing Values

In [ ]:
# ── Generate Dataset with 5% Missing Values ───────────────────────────────────
np.random.seed(42)
n_samples = 1200

# Feature generation
sqft          = np.random.normal(1800, 500, n_samples).clip(500, 5000)
bedrooms      = np.random.choice([2, 3, 4, 5], n_samples, p=[0.20, 0.45, 0.25, 0.10])
bathrooms     = np.random.choice([1, 2, 3, 4], n_samples, p=[0.15, 0.50, 0.25, 0.10])
age           = np.random.randint(0, 60, n_samples).astype(float)
school_dist   = np.random.uniform(0, 10, n_samples)
garage        = np.random.choice([0, 1, 2], n_samples, p=[0.15, 0.40, 0.45]).astype(float)
renovated     = np.random.choice([0, 1], n_samples, p=[0.65, 0.35]).astype(float)
crime_rate    = np.random.uniform(0, 10, n_samples)
lot_size      = np.random.normal(8000, 3000, n_samples).clip(2000, 25000)
hoa_fee       = np.random.uniform(0, 600, n_samples)   # HOA fee — often missing

# Price formula (nonlinear, includes interactions)
price = (
    120 * sqft / 1000
    + 15 * bedrooms
    + 18 * bathrooms
    - 1.2 * age
    - 8   * school_dist
    + 12  * garage
    + 25  * renovated
    - 5   * crime_rate
    + 0.002 * lot_size
    - 0.05  * hoa_fee
    + 0.002 * sqft * renovated    # interaction term
    + np.random.normal(0, 15, n_samples)
) * 1000

# Build DataFrame
df = pd.DataFrame({
    'sqft':        sqft,
    'bedrooms':    bedrooms.astype(float),
    'bathrooms':   bathrooms.astype(float),
    'age':         age,
    'school_dist': school_dist,
    'garage':      garage,
    'renovated':   renovated,
    'crime_rate':  crime_rate,
    'lot_size':    lot_size,
    'hoa_fee':     hoa_fee,
    'price':       price
})

# Inject 5% missing values (MAR — missing at random) into features
feature_cols = [c for c in df.columns if c != 'price']
missing_rate = 0.05
rng = np.random.default_rng(42)
for col in feature_cols:
    mask = rng.random(n_samples) < missing_rate
    df.loc[mask, col] = np.nan

total_missing = df[feature_cols].isna().sum().sum()
total_cells   = n_samples * len(feature_cols)

print(f"Dataset shape: {df.shape}")
print(f"Total missing values: {total_missing} / {total_cells} cells = {total_missing/total_cells*100:.1f}%")
print(f"\nMissing per feature:")
print(df[feature_cols].isna().sum().to_string())
print(f"\nPrice range: ${df['price'].min()/1000:.0f}K – ${df['price'].max()/1000:.0f}K")

In [ ]:
# ── Train/Val Split ───────────────────────────────────────────────────────────
X = df[feature_cols].values
y = df['price'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Also prepare imputed version (for sklearn GB which can't handle NaN)
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)

print(f"Train: {X_train.shape[0]} samples  |  Val: {X_val.shape[0]} samples")
print(f"Missing in X_train: {np.isnan(X_train).sum():.0f} cells")
print(f"Missing in X_val:   {np.isnan(X_val).sum():.0f} cells")
print(f"Imputed versions: no NaN  ✓")

## Section 3: Sklearn GradientBoostingRegressor vs. XGBoost Baseline

In [ ]:
# ── Baseline Comparison: sklearn GB vs. XGBoost ──────────────────────────────
results = {}

# 1. Sklearn GradientBoostingRegressor (needs imputed data)
t0 = time.time()
gb_sk = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.1, max_depth=4,
    subsample=1.0, random_state=42
)
gb_sk.fit(X_train_imp, y_train)
t_sk = time.time() - t0

rmse_sk_train = np.sqrt(mean_squared_error(y_train, gb_sk.predict(X_train_imp)))
rmse_sk_val   = np.sqrt(mean_squared_error(y_val,   gb_sk.predict(X_val_imp)))
results['sklearn GB'] = {'train_rmse': rmse_sk_train, 'val_rmse': rmse_sk_val, 'time': t_sk}

# 2. XGBoost (handles NaN natively)
t0 = time.time()
gb_xgb = xgb.XGBRegressor(
    n_estimators=200, learning_rate=0.1, max_depth=4,
    subsample=1.0, colsample_bytree=1.0,
    reg_lambda=1.0, reg_alpha=0.0, gamma=0.0,
    random_state=42,
    verbosity=0
)
gb_xgb.fit(X_train, y_train)    # raw X_train with NaN — XGBoost handles it!
t_xgb = time.time() - t0

rmse_xgb_train = np.sqrt(mean_squared_error(y_train, gb_xgb.predict(X_train)))
rmse_xgb_val   = np.sqrt(mean_squared_error(y_val,   gb_xgb.predict(X_val)))
results['XGBoost'] = {'train_rmse': rmse_xgb_train, 'val_rmse': rmse_xgb_val, 'time': t_xgb}

# Summary table
print("=" * 62)
print(f"{'Model':<22} {'Train RMSE':>12} {'Val RMSE':>12} {'Time (s)':>10}")
print("-" * 62)
for name, r in results.items():
    print(f"{name:<22} ${r['train_rmse']:>10,.0f} ${r['val_rmse']:>10,.0f} {r['time']:>9.2f}s")
print("=" * 62)
print()
speedup = results['sklearn GB']['time'] / results['XGBoost']['time']
print(f"XGBoost speedup: {speedup:.1f}x faster than sklearn GB")
improvement = (results['sklearn GB']['val_rmse'] - results['XGBoost']['val_rmse'])
print(f"XGBoost val RMSE improvement: ${improvement:,.0f}")

## Section 4: Missing Value Handling — XGBoost's Key Advantage

XGBoost learns the **optimal default direction** for missing values at each split node. During training, it tries both left and right directions for missing values and picks the one that reduces the objective more.

In [ ]:
# ── Missing Value Handling Comparison ────────────────────────────────────────
# We test at increasing missing rates to stress-test each approach
missing_rates = [0.0, 0.05, 0.10, 0.20, 0.35, 0.50]

rmse_xgb_native = []
rmse_gb_imputed = []

for rate in missing_rates:
    np.random.seed(42)

    # Inject missing values at the specified rate
    X_tr_miss = X_train.copy().astype(float)
    X_vl_miss = X_val.copy().astype(float)
    for arr in [X_tr_miss, X_vl_miss]:
        mask = np.random.random(arr.shape) < rate
        arr[mask] = np.nan

    # XGBoost: pass data with NaN directly
    if XGB_AVAILABLE:
        m_xgb = xgb.XGBRegressor(
            n_estimators=200, learning_rate=0.1, max_depth=4,
            random_state=42, verbosity=0
        )
        m_xgb.fit(X_tr_miss, y_train)
        rmse_xgb_native.append(
            np.sqrt(mean_squared_error(y_val, m_xgb.predict(X_vl_miss)))
        )
    else:
        # Fallback: use imputation as stand-in
        imp = SimpleImputer(strategy='median')
        m_xgb = GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42
        )
        m_xgb.fit(imp.fit_transform(X_tr_miss), y_train)
        rmse_xgb_native.append(
            np.sqrt(mean_squared_error(y_val, m_xgb.predict(imp.transform(X_vl_miss))))
        )

    # Sklearn GB: requires median imputation
    imp = SimpleImputer(strategy='median')
    m_gb = GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42
    )
    m_gb.fit(imp.fit_transform(X_tr_miss), y_train)
    rmse_gb_imputed.append(
        np.sqrt(mean_squared_error(y_val, m_gb.predict(imp.transform(X_vl_miss))))
    )

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
pct_labels = [f'{r*100:.0f}%' for r in missing_rates]
x = np.arange(len(missing_rates))

label_xgb = 'XGBoost (native NaN)' if XGB_AVAILABLE else 'XGBoost fallback (imputed)'
ax.plot(x, np.array(rmse_xgb_native)/1000, 'o-', color='darkorange',
        lw=2, ms=7, label=label_xgb)
ax.plot(x, np.array(rmse_gb_imputed)/1000, 's--', color='steelblue',
        lw=2, ms=7, label='sklearn GB (median imputation)')

ax.set_xticks(x)
ax.set_xticklabels(pct_labels)
ax.set_xlabel('Missing Data Rate')
ax.set_ylabel('Validation RMSE ($K)')
ax.set_title('Missing Value Handling: XGBoost Native vs. Imputation', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

print("\nKey insight:")
print("  XGBoost's native NaN handling LEARNS the optimal routing for missing values.")
print("  Median imputation forces a single global estimate — loses directional info.")
print("  At 50% missing rate, the gap between the two approaches is most visible.")

## Section 5: Second-Order Information — Gradient vs. Hessian

This illustration shows why using curvature (Hessian) gives better step sizes. On a flat loss landscape, large steps are safe. On a curved landscape, the Hessian tells you to slow down.

In [ ]:
# ── Second-Order Illustration: Gradient vs. Gradient+Hessian ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Why Second-Order Information Gives Better Steps',
             fontsize=12, fontweight='bold')

F_range = np.linspace(-3, 5, 300)

# --- Panel 1: MSE loss (parabola) — constant Hessian ---
ax = axes[0]
y_true_ex = 2.0  # True value
L_mse = 0.5 * (y_true_ex - F_range)**2
F_cur = -1.0  # Current prediction

g_mse = F_cur - y_true_ex       # gradient: predicted - actual
h_mse = 1.0                      # Hessian (constant for MSE)
F_next_1st = F_cur - g_mse       # first-order step (full Newton = same for MSE)
F_next_2nd = F_cur - g_mse / h_mse  # second-order step

ax.plot(F_range, L_mse, 'k-', lw=2, label='L(F) = MSE')
ax.axvline(y_true_ex, color='gray', ls='--', lw=1, label=f'Minimum (F*={y_true_ex})')
ax.scatter([F_cur], [0.5*(y_true_ex-F_cur)**2],
           s=100, color='crimson', zorder=5, label=f'Current F={F_cur}')
ax.scatter([F_next_2nd], [0.5*(y_true_ex-F_next_2nd)**2],
           s=100, color='steelblue', marker='*', zorder=5, label=f'XGBoost step → F={F_next_2nd:.1f}')
ax.set_xlabel('F (prediction)')
ax.set_ylabel('Loss')
ax.set_title('MSE Loss\n(Hessian=1, constant curvature)')
ax.legend(fontsize=7)

# --- Panel 2: Log-loss (non-constant Hessian) ---
ax = axes[1]
y_bin = 1.0  # True binary label
sigma = lambda f: 1 / (1 + np.exp(-f))
L_log = -(y_bin * np.log(sigma(F_range) + 1e-10)
          + (1-y_bin) * np.log(1 - sigma(F_range) + 1e-10))

F_cur_log  = -2.0
p          = sigma(F_cur_log)
g_log      = p - y_bin           # gradient
h_log      = p * (1 - p)         # Hessian (varies with prediction!)

F_step_1st = F_cur_log - 0.5 * g_log            # first-order with lr=0.5
F_step_2nd = F_cur_log - g_log / (h_log + 1e-6)  # second-order (Newton)

ax.plot(F_range, L_log, 'k-', lw=2, label='L(F) = Log-loss')
ax.axvline(0, color='gray', ls='--', lw=1, label='Minimum (F*=0 for y=1 approx)')
ax.scatter([F_cur_log], [-(y_bin*np.log(p+1e-10)+(1-y_bin)*np.log(1-p+1e-10))],
           s=100, color='crimson', zorder=5, label=f'Current F={F_cur_log}')
ax.scatter([F_step_1st], [np.interp(F_step_1st, F_range, L_log)],
           s=80, color='orange', marker='D', zorder=5, label=f'1st-order step → {F_step_1st:.1f}')
ax.scatter([F_step_2nd], [np.interp(F_step_2nd, F_range, L_log)],
           s=100, color='steelblue', marker='*', zorder=5, label=f'2nd-order step → {F_step_2nd:.1f}')
ax.set_xlabel('F (log-odds)')
ax.set_title('Log-Loss\n(Hessian=p(1-p), varies with prediction)')
ax.legend(fontsize=7)

# --- Panel 3: Hessian value along F for log-loss ---
ax = axes[2]
h_vals = sigma(F_range) * (1 - sigma(F_range))
ax.plot(F_range, h_vals, color='darkorange', lw=2, label='h(F) = σ(F)·(1−σ(F))')
ax.axvline(F_cur_log, color='crimson', ls='--', lw=1.5, label=f'Current F={F_cur_log}')
ax.scatter([F_cur_log], [h_log], s=100, color='crimson', zorder=5)
ax.set_xlabel('F (log-odds)')
ax.set_ylabel('Hessian value')
ax.set_title('Hessian (Curvature) of Log-Loss\n(Low at extremes → larger steps safe)')
ax.legend(fontsize=8)
ax.text(0.03, 0.90,
        f'At F={F_cur_log}: h={h_log:.3f}\n→ step = {g_log:.3f}/{h_log:.3f} = {g_log/h_log:.2f}',
        transform=ax.transAxes, fontsize=8,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print("\nKey insight:")
print("  For MSE, the Hessian is always 1 → second-order steps = first-order steps.")
print("  For log-loss, the Hessian VARIES: low at extreme predictions (|F|>>0).")
print("  XGBoost uses h_i to scale step sizes — bigger steps where curvature is low.")
print("  This is why XGBoost outperforms vanilla GB on classification tasks.")

## Section 6: Regularization — Lambda and Gamma Effects

In [ ]:
# ── Regularization Grid: reg_lambda × gamma ────────────────────────────────
if XGB_AVAILABLE:
    lambdas = [0.0, 0.1, 1.0, 10.0]
    gammas  = [0.0, 0.1, 1.0]

    val_rmse_grid = np.zeros((len(lambdas), len(gammas)))

    for i, lam in enumerate(lambdas):
        for j, gam in enumerate(gammas):
            m = xgb.XGBRegressor(
                n_estimators=200, learning_rate=0.1, max_depth=4,
                reg_lambda=lam, gamma=gam, reg_alpha=0.0,
                subsample=1.0, colsample_bytree=1.0,
                random_state=42, verbosity=0
            )
            m.fit(X_train, y_train)
            val_rmse_grid[i, j] = np.sqrt(mean_squared_error(y_val, m.predict(X_val))) / 1000

    grid_df = pd.DataFrame(
        val_rmse_grid,
        index=[f'λ={l}' for l in lambdas],
        columns=[f'γ={g}' for g in gammas]
    )

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(grid_df, annot=True, fmt='.2f', cmap='RdYlGn_r',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'Val RMSE ($K)'})
    ax.set_title('XGBoost Regularization Grid\nVal RMSE ($K) — reg_lambda × gamma', fontweight='bold')
    ax.set_xlabel('γ (minimum split gain threshold)')
    ax.set_ylabel('λ (L2 leaf weight regularization)')
    plt.tight_layout()
    plt.show()

    best_idx = np.unravel_index(np.argmin(val_rmse_grid), val_rmse_grid.shape)
    print(f"Best combination: λ={lambdas[best_idx[0]]}, γ={gammas[best_idx[1]]}")
    print(f"Best val RMSE: ${val_rmse_grid[best_idx]*1000:,.0f}")

else:
    print("XGBoost not installed — showing sklearn GB subsample regularization instead.")
    subsamples = [0.3, 0.5, 0.8, 1.0]
    depths     = [2, 3, 5]
    grid_data  = []
    for d in depths:
        row = []
        for s in subsamples:
            m = GradientBoostingRegressor(
                n_estimators=200, learning_rate=0.1, max_depth=d,
                subsample=s, random_state=42
            )
            m.fit(X_train_imp, y_train)
            row.append(np.sqrt(mean_squared_error(y_val, m.predict(X_val_imp))) / 1000)
        grid_data.append(row)

    gdf = pd.DataFrame(grid_data, index=[f'depth={d}' for d in depths],
                        columns=[f'sub={s}' for s in subsamples])
    fig, ax = plt.subplots(figsize=(6, 3.5))
    sns.heatmap(gdf, annot=True, fmt='.1f', cmap='RdYlGn_r', linewidths=0.5, ax=ax)
    ax.set_title('sklearn GB: Depth × Subsample — Val RMSE ($K)')
    plt.tight_layout()
    plt.show()

## Section 7: Column Subsampling (colsample_bytree)

Like random forests, XGBoost can randomly select a fraction of features for each tree. This reduces correlation between trees and acts as regularization.

In [ ]:
# ── Column Subsampling Effect ─────────────────────────────────────────────────
colsample_vals = [0.3, 0.5, 0.8, 1.0]
colors         = ['purple', 'tomato', 'steelblue', 'forestgreen']
n_est          = 300

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

overfitting_gaps = []

for cs, color in zip(colsample_vals, colors):
    if XGB_AVAILABLE:
        m = xgb.XGBRegressor(
            n_estimators=n_est, learning_rate=0.05, max_depth=5,
            colsample_bytree=cs, subsample=1.0,
            random_state=42, verbosity=0
        )
        m.fit(X_train, y_train,
              eval_set=[(X_train, y_train), (X_val, y_val)],
              verbose=False)

        # Extract staged results from evals_result_
        er         = m.evals_result()
        tr_key     = list(er.keys())[0]
        vl_key     = list(er.keys())[1] if len(er) > 1 else list(er.keys())[0]
        train_rmse = np.array(er[tr_key].get('rmse', er[tr_key].get('train-rmse', [0])))
        val_rmse   = np.array(er[vl_key].get('rmse', er[vl_key].get('validation-rmse', [0])))

        # If evals_result keys differ, re-collect with staged predict fallback
        if len(train_rmse) < 2:
            train_rmse = np.array([np.sqrt(mean_squared_error(y_train, m.predict(X_train)))])
            val_rmse   = np.array([np.sqrt(mean_squared_error(y_val,   m.predict(X_val)))])
    else:
        m = GradientBoostingRegressor(
            n_estimators=n_est, learning_rate=0.05, max_depth=5,
            subsample=cs, random_state=42
        )
        m.fit(X_train_imp, y_train)
        train_rmse = np.array([np.sqrt(mean_squared_error(y_train, p))
                                for p in m.staged_predict(X_train_imp)])
        val_rmse   = np.array([np.sqrt(mean_squared_error(y_val, p))
                                for p in m.staged_predict(X_val_imp)])

    label      = f'colsample={cs}'
    stages     = np.arange(len(train_rmse))
    overfitting_gaps.append(train_rmse[-1] - val_rmse[-1])

    axes[0].plot(stages, train_rmse/1000, color=color, lw=1.5, label=label)
    axes[1].plot(stages, val_rmse/1000,   color=color, lw=1.5, label=label)

    best = np.argmin(val_rmse)
    axes[1].scatter(stages[best], val_rmse[best]/1000, color=color, s=80, zorder=5)

for ax, title in [(axes[0], 'Train RMSE ($K)'), (axes[1], 'Validation RMSE ($K)')]:
    ax.set_xlabel('Number of Trees')
    ax.set_ylabel('RMSE ($K)')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Column Subsampling: colsample_bytree Effect on Overfitting',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nOverfitting gap (train RMSE - val RMSE) at last tree:")
for cs, gap in zip(colsample_vals, overfitting_gaps):
    print(f"  colsample_bytree={cs}: ${abs(gap)/1000:.1f}K gap")

## Section 8: Early Stopping

Instead of guessing the right number of trees, XGBoost can automatically stop when the validation metric stops improving. This saves both tuning time and computation.

In [ ]:
# ── Early Stopping Demo ───────────────────────────────────────────────────────
if XGB_AVAILABLE:
    model_es = xgb.XGBRegressor(
        n_estimators=1000,          # set high — early stopping will cut this down
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        early_stopping_rounds=15,   # stop if val RMSE doesn't improve for 15 rounds
        eval_metric='rmse',
        random_state=42,
        verbosity=0
    )

    model_es.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False
    )

    best_iter  = model_es.best_iteration
    er         = model_es.evals_result()
    keys       = list(er.keys())

    # Extract RMSE curves from evals_result — key names vary by xgboost version
    def _get_rmse(er_dict):
        for k in ['rmse', 'validation-rmse', 'train-rmse']:
            if k in er_dict:
                return np.array(er_dict[k])
        return np.array(list(er_dict.values())[0])

    train_rmse_es = _get_rmse(er[keys[0]])
    val_rmse_es   = _get_rmse(er[keys[-1]])

    fig, ax = plt.subplots(figsize=(9, 4))
    stages = np.arange(len(train_rmse_es))

    ax.plot(stages, train_rmse_es/1000, color='steelblue', lw=2, label='Train RMSE')
    ax.plot(stages, val_rmse_es/1000,   color='darkorange', lw=2, label='Val RMSE')

    # Mark the best iteration and early stop point
    ax.axvline(best_iter, color='crimson', ls='--', lw=1.5,
               label=f'Best iteration: {best_iter}')
    ax.axvline(len(stages)-1, color='gray', ls=':', lw=1.5,
               label=f'Early stop at: {len(stages)-1}')
    ax.scatter([best_iter], [val_rmse_es[best_iter]/1000],
               s=150, color='crimson', zorder=6, marker='*')

    # Shade the "wasted" region past best iteration
    ax.axvspan(best_iter, len(stages)-1, alpha=0.08, color='red',
               label='Post-optimal region')

    ax.set_xlabel('Number of Trees')
    ax.set_ylabel('RMSE ($K)')
    ax.set_title('XGBoost Early Stopping: Automatically Finds Optimal n_estimators',
                 fontweight='bold')
    ax.legend(fontsize=9)

    best_val_rmse = val_rmse_es[best_iter]
    ax.text(0.62, 0.95,
            f'Best val RMSE: ${best_val_rmse/1000:.1f}K\n'
            f'At tree #{best_iter}\n'
            f'Stopped at tree #{len(stages)-1}\n'
            f'(patience = 15 rounds)',
            transform=ax.transAxes, fontsize=9, va='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85))

    plt.tight_layout()
    plt.show()

    print(f"\nEarly stopping result:")
    print(f"  Requested up to 1000 trees")
    print(f"  Optimal trees found:  {best_iter}")
    print(f"  Stopped training at:  {len(stages)-1} trees")
    print(f"  Best val RMSE:        ${best_val_rmse:,.0f}")
    print(f"  Trees saved by early stopping: {1000 - len(stages)}")

else:
    print("XGBoost not installed. Demonstrating early stopping with staged_predict.")

    model_sk = GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=4,
        subsample=0.8, random_state=42
    )
    model_sk.fit(X_train_imp, y_train)

    val_rmses = [np.sqrt(mean_squared_error(y_val, p))
                 for p in model_sk.staged_predict(X_val_imp)]
    tr_rmses  = [np.sqrt(mean_squared_error(y_train, p))
                 for p in model_sk.staged_predict(X_train_imp)]

    best_iter = int(np.argmin(val_rmses))

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(tr_rmses, color='steelblue', lw=2, label='Train RMSE')
    ax.plot(val_rmses, color='darkorange', lw=2, label='Val RMSE')
    ax.axvline(best_iter, color='crimson', ls='--', lw=1.5,
               label=f'Best stage: {best_iter}')
    ax.scatter([best_iter], [val_rmses[best_iter]/1000],
               s=150, color='crimson', zorder=6, marker='*')
    ax.set_xlabel('Number of Trees')
    ax.set_ylabel('RMSE')
    ax.set_title('sklearn GB: Finding Optimal n_trees via staged_predict', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"Best stage: {best_iter} trees | Val RMSE: ${val_rmses[best_iter]:,.0f}")

## Section 9: XGBoost Hyperparameter Reference & Tuning Strategy

In [ ]:
# ── Feature Importance (if XGBoost available) ─────────────────────────────────
if XGB_AVAILABLE:
    final_model = xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        reg_lambda=1.0, reg_alpha=0.0,
        random_state=42, verbosity=0
    )
    final_model.fit(X_train, y_train)

    importances = final_model.feature_importances_
    feat_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
    feat_df = feat_df.sort_values('importance', ascending=True)

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.barh(feat_df['feature'], feat_df['importance'],
                   color=sns.color_palette('viridis', len(feature_cols)))
    ax.set_xlabel('Feature Importance (gain)')
    ax.set_title('XGBoost Feature Importances\n(Gain: average loss reduction per split)',
                 fontweight='bold')

    # Add value labels
    for bar, val in zip(bars, feat_df['importance']):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)

    plt.tight_layout()
    plt.show()

    final_val_rmse = np.sqrt(mean_squared_error(y_val, final_model.predict(X_val)))
    print(f"Final model val RMSE: ${final_val_rmse:,.0f}")
    print(f"\nTop 3 most important features:")
    for _, row in feat_df.tail(3).iloc[::-1].iterrows():
        print(f"  {row['feature']}: {row['importance']:.4f}")

else:
    # sklearn GB feature importances
    final_model = GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        subsample=0.8, random_state=42
    )
    final_model.fit(X_train_imp, y_train)
    importances = final_model.feature_importances_
    feat_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
    feat_df = feat_df.sort_values('importance', ascending=True)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(feat_df['feature'], feat_df['importance'],
            color=sns.color_palette('viridis', len(feature_cols)))
    ax.set_xlabel('Feature Importance')
    ax.set_title('sklearn GB Feature Importances', fontweight='bold')
    plt.tight_layout()
    plt.show()

## Section 10: Hyperparameter Cheat Sheet & Tuning Strategy

### XGBoost Key Hyperparameters

| Parameter | Typical Range | Effect |
|:---|:---:|:---|
| `n_estimators` | 100–3000 | Use early stopping — don't tune manually |
| `learning_rate` | 0.01–0.3 | Smaller → more trees → better generalization |
| `max_depth` | 3–8 | Deeper → more interactions → more overfit |
| `subsample` | 0.5–1.0 | Row subsampling — reduces overfit |
| `colsample_bytree` | 0.5–1.0 | Column subsampling per tree |
| `reg_lambda` | 0.1–10 | L2 on leaf weights — primary regularizer |
| `reg_alpha` | 0–1 | L1 on leaf weights — drives sparsity |
| `gamma` | 0–5 | Min split gain threshold — implicit pruning |
| `min_child_weight` | 1–20 | Min Hessian sum in leaf (≡ min_samples_leaf for MSE) |

### Recommended Tuning Order (Practical Guide)

1. **Start:** `learning_rate=0.1`, `max_depth=4`, `n_estimators=500` with early stopping
2. **Tune depth:** try `max_depth` ∈ {3, 4, 5, 6} — most impactful single parameter
3. **Add subsampling:** try `subsample=0.8`, `colsample_bytree=0.8`
4. **Tune regularization:** `reg_lambda` ∈ {0.1, 1, 5, 10}
5. **Lower learning rate:** halve `learning_rate`, double early stopping patience
6. **Final fit:** retrain with best params + early stopping on full train set

### XGBoost vs. Gradient Boosting vs. Random Forest

| Property | Random Forest | sklearn GB | XGBoost |
|:---|:---:|:---:|:---:|
| Second-order opt. | No | No | Yes |
| Built-in L1/L2 | No | No | Yes |
| NaN handling | No | No | Yes |
| Parallel training | Easy | Hard | Partial (within-level) |
| Typical accuracy | Good | Better | Best |
| Training speed | Fast | Slow | Fast (histogram) |
| Tuning complexity | Low | Medium | Medium |

In [ ]:
# ── Final Model Comparison Summary ────────────────────────────────────────────
print("FINAL COMPARISON: All Models on House Price Dataset")
print("=" * 65)
print(f"{'Model':<35} {'Val RMSE':>12} {'Notes':>15}")
print("-" * 65)

# Baseline: predict mean
baseline_rmse = np.sqrt(mean_squared_error(y_val, np.full_like(y_val, y_train.mean())))
print(f"{'Baseline (predict mean)':<35} ${baseline_rmse:>10,.0f}")

# sklearn GB
rmse_gb = np.sqrt(mean_squared_error(y_val, gb_sk.predict(X_val_imp)))
print(f"{'sklearn GradientBoostingRegressor':<35} ${rmse_gb:>10,.0f} {'(imputed)'}>")

# XGBoost
if XGB_AVAILABLE:
    rmse_xgb = np.sqrt(mean_squared_error(y_val, final_model.predict(X_val)))
    print(f"{'XGBoost (tuned)':<35} ${rmse_xgb:>10,.0f} {'(native NaN)'}")
else:
    print("  XGBoost not available — see installation instructions above.")

print("=" * 65)
print()
print("Key takeaways:")
print("  1. XGBoost's second-order optimization improves split decisions.")
print("  2. Regularization (λ, γ) is built into the objective — not post-hoc.")
print("  3. Native NaN handling eliminates the imputation pipeline entirely.")
print("  4. Early stopping eliminates the need to tune n_estimators manually.")
print("  5. colsample_bytree adds diversity à la random forests.")

## Self-Check Questions

Test your understanding. Try to answer before revealing the solution.

---

**Question 1:** XGBoost adds a γ parameter (minimum split gain). When γ=0, every split that reduces loss is accepted. When γ=10, only large improvements are accepted. How is this related to cost-complexity pruning in CART decision trees?

<details>
<summary>Click to reveal answer</summary>

**Answer:**

They are mathematically equivalent in spirit. Recall CART's cost-complexity criterion:
$$R_\alpha(T) = \sum_{\text{leaves}} R(t) + \alpha |T|$$

A split is only retained in post-pruning if the improvement in training error exceeds the $\alpha$ penalty per additional leaf.

In XGBoost, the gain formula is:
$$\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{G^2}{H+\lambda}\right] - \gamma$$

A split is only made if Gain > 0, meaning the impurity reduction must exceed $\gamma$. So:
- $\gamma$ plays the same role as $\alpha$ in CART — it is the **minimum improvement threshold** to accept a split
- Both are **complexity penalties** per split
- The key difference: CART does cost-complexity **post-pruning** (grow then prune), while XGBoost uses $\gamma$ during **tree construction** (pre-pruning)

Larger $\gamma$ → more aggressive pruning → simpler trees → stronger regularization.

</details>

---

**Question 2:** XGBoost uses Hessians (second derivatives) to compute optimal leaf weights. For MSE loss: $g_i = \hat{y}_i - y_i$ and $h_i = 1$ (constant). What does this mean for the optimal leaf weight formula, and why is this intuitive?

<details>
<summary>Click to reveal answer</summary>

**Answer:**

The optimal leaf weight formula is:
$$w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda} = -\frac{\sum_{i \in I_j}(\hat{y}_i - y_i)}{|I_j| + \lambda}$$

Since $h_i = 1$ for all samples with MSE:
$$w_j^* = \frac{\sum_{i \in I_j}(y_i - \hat{y}_i)}{|I_j| + \lambda} = \frac{\text{sum of residuals in leaf}}{\text{leaf size} + \lambda}$$

This is simply the **regularized mean residual** in that leaf — exactly what a standard regression tree would predict as the leaf value, but with an extra $\lambda$ in the denominator that shrinks the prediction toward zero.

**Why intuitive:** For MSE, the best prediction for a group of samples is always their mean. XGBoost's formula reduces to this mean, plus a regularization term $\lambda$ that prevents extreme leaf weights when a leaf has few samples.

This explains why XGBoost and standard gradient boosting give nearly identical results for MSE regression: for MSE, the Hessian adds no new directional information (it's constant), so the second-order advantage only materializes for other loss functions like log-loss.

</details>

---

**Question 3:** XGBoost's `min_child_weight` requires the sum of Hessians in a leaf to be ≥ `min_child_weight`. For MSE loss (where $h_i = 1$), this is equivalent to what sklearn parameter?

<details>
<summary>Click to reveal answer</summary>

**Answer:**

For MSE loss, $h_i = 1$ for every sample. Therefore:
$$\sum_{i \in I_j} h_i = \sum_{i \in I_j} 1 = |I_j| \quad (\text{the number of samples in leaf } j)$$

So `min_child_weight` ≥ k means "leaf must contain at least k samples" — which is exactly sklearn's **`min_samples_leaf`** parameter.

For other loss functions:
- **Log-loss:** $h_i = p_i(1-p_i)$ varies between 0 and 0.25. `min_child_weight=1` means the leaf must have enough samples such that the sum of $p_i(1-p_i)$ ≥ 1. For well-calibrated probabilities near 0.5, this is ~4 samples; for confident predictions near 0 or 1, it may require more samples.
- This makes `min_child_weight` **adaptive**: it automatically requires more samples in leaves with uncertain predictions, and fewer in leaves with confident predictions.

This is one reason XGBoost's `min_child_weight` is more principled than a fixed `min_samples_leaf` for classification tasks.

</details>